In [ ]:
from pathlib import Path

import numpy  as np
import pandas as pd
import tables as tb
import matplotlib.pyplot as plt

from nbtools import normhist
from nbtools import normhist2d
from nbtools import auto_plot_style
from jupyfun import progressbar

from invisible_cities.core.core_functions import in_range
from invisible_cities.io  . dst_io        import df_writer

In [ ]:
%matplotlib inline

auto_plot_style()

In [ ]:
def positive_drift(df):
    return df.DT>0

In [ ]:
run = XXXXX

In [ ]:
folder     = Path(f"/media/gonzalo/se/data/NEXT/next100/{run}-kr/sophronia/")
folder     = sorted(folder.glob("trigger*"))[0]
filenames  = sorted(folder.glob("*.hdst"))
filename   = filenames[0]
outputfile = filename.parent / "selected_events.kdst"

data = pd.read_hdf(filename, "/DST/Events")
data = data.loc[positive_drift]
print(len(data), data.event.nunique())

In [ ]:
df = data.groupby("event").first()
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
normhist(data.nS1, np.arange(50))
plt.xlabel("# S1 peaks")
plt.ylabel("Frequency (%)")
plt.grid()

plt.subplot(1, 2, 2)
normhist(data.nS2, np.arange(50))
plt.xlabel("# S2 peaks")
plt.ylabel("Frequency (%)")
plt.grid()

plt.tight_layout()

In [ ]:
dtbins     = np.linspace(0, 1600, 101)
dtrmsbins  = np.linspace(0, 10, 101)
dtrms2bins = np.linspace(0, 55, 101)
ebins      = np.linspace(0, 15e3, 101)

In [ ]:
df = data
normhist2d(df.DT, df.Zrms, (dtbins, dtrmsbins), cmin=1e-3)
plt.xlabel("Drift time ($\mu$s)")
plt.ylabel("DT-rms ($\mu$s)")
plt.grid()

In [ ]:
df = data
normhist2d(df.DT, df.Zrms**2, (dtbins, dtrms2bins), cmin=1e-3)
dtrms2_low = lambda dt: -0.7 + 0.030 * (dt-20)
plt.plot(df.DT, dtrms2_low(df.DT), ".r", ms=2)
dtrms2_upp = lambda dt: 2.6 + 0.036 * (dt-20)
plt.plot(df.DT, dtrms2_upp(df.DT), ".r", ms=2)
plt.xlabel("Drift time ($\mu$s)")
plt.ylabel("DT$_{rms}^2$ ($\mu$s)")
plt.grid()

In [ ]:
def in_band(df):
    return in_range(df.Zrms**2, dtrms2_low(df.DT), dtrms2_upp(df.DT))

In [ ]:
df = data.loc[in_band]
print(len(df), df.event.nunique(), df.event.nunique()/data.event.nunique())

normhist2d(df.DT, df.Zrms, 100, cmin=1e-6)
plt.xlabel("Drift time ($\mu$s)")
plt.ylabel("DT-rms ($\mu$s)")
plt.grid()

plt.figure(figsize=(15, 6))
normhist(df.DT, np.linspace(0, 1500, 101))
plt.xlabel("Drift time ($\mu$s)")
plt.ylabel("Frequency (%)")
plt.grid()

plt.figure()
normhist(df.S2e, ebins)
plt.xlabel("S2 energy (pe)")
plt.ylabel("Frequency (%)")
plt.grid()

plt.figure()
normhist2d(df.DT, df.S2e, (dtbins, ebins), cmin=1e-6)
s2e_upp = lambda dt: 9500 - 0.30*dt
plt.plot(dtbins, s2e_upp(dtbins), "r-")
plt.xlabel("Drift time ($\mu$s)")
plt.ylabel("S2 energy (pe)")
plt.grid()

In [ ]:
def kr_energy(df):
    return in_range(df.S2e, 0, s2e_upp(df.DT))

In [ ]:
df = data.loc[in_band].loc[kr_energy]
print(len(df), df.event.nunique(), df.event.nunique()/data.event.nunique())
normhist(df.DT, dtbins)
plt.xlabel("Drift time ($\mu$s)")
plt.ylabel("Frequency (%)")

In [ ]:
def recompute_npeaks(df):
    events = df.groupby("event")
    ns1    = events.s1_peak.nunique().values
    ns2    = events.s2_peak.nunique().values
    n      = events.s1_peak.count  ().values
    df.loc[:, "nS1"] = np.repeat(ns1, n)
    df.loc[:, "nS2"] = np.repeat(ns2, n)
    return df

In [ ]:
df = data.loc[in_band].loc[kr_energy]
print(len(df), df.event.nunique(), df.event.nunique()/data.event.nunique())

In [ ]:
data = []
effs = []
for i, filename in progressbar(filenames, index=True):
    assert filename.exists()
    eff = dict()
    df = pd.read_hdf(filename, "/DST/Events"); eff["n_kdst"] = df.event.nunique()
    df = df.loc[positive_drift]; eff["n_reco"  ] = df.event.nunique()
    df = df.loc[in_band       ]; eff["n_band"  ] = df.event.nunique()
    df = df.loc[kr_energy     ]; eff["n_energy"] = df.event.nunique()
    data.append(df.reset_index())
    effs.append(pd.DataFrame(eff, index=[i+1]))

data = pd.concat(data, ignore_index=True)
data.sort_values("event s2_peak s1_peak".split(), inplace=True)
data = recompute_npeaks(data)
effs = pd.concat(effs).sum()
effs

In [ ]:
df = data.groupby("event").first()

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
normhist(df.nS1, np.arange(16)  - 0.5, histtype="step")
plt.xlabel("nS1")
plt.ylabel("Frequency (%)")
plt.grid()

plt.subplot(1, 2, 2)
normhist(df.nS2, np.arange(8), histtype="step")
plt.xlabel("nS2")
plt.ylabel("Frequency (%)")
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize=(30, 12))

df = data.groupby("event s1_peak".split()).first()

plt.subplot(2, 4, 1)
normhist(df.S1w, np.arange(0, 500, 25), histtype="step")
plt.xlabel("S1 width (ns)")
plt.ylabel("Fraction of events (%)")
plt.grid()

plt.subplot(2, 4, 2)
normhist(df.S1h, np.linspace(0, 5, 51), histtype="step")
plt.xlabel("S1 height (pes)")
plt.ylabel("Fraction of events (%)")
plt.grid()

plt.subplot(2, 4, 3)
normhist(df.S1e, np.linspace(0, 30, 51), histtype="step")
plt.xlabel("S1 energy (pes)")
plt.ylabel("Fraction of events (%)")
plt.grid()

df = data.groupby("event s2_peak".split()).first()

plt.subplot(2, 4, 5)
normhist(df.S2w, np.arange(50), histtype="step")
plt.xlabel("S2 width ($\mu$s)")
plt.ylabel("Fraction of events (%)")
plt.grid()

plt.subplot(2, 4, 6)
normhist(df.S2h, np.linspace(0, 2000, 51), histtype="step")
plt.xlabel("S2 height (pes)")
plt.ylabel("Fraction of events (%)")
plt.grid()

plt.subplot(2, 4, 7)
normhist(df.S2e, np.linspace(0, 12e3, 101), histtype="step")
plt.xlabel("S2 energy (pes)")
plt.ylabel("Fraction of events (%)")
plt.grid()

plt.subplot(2, 4, 8)
normhist(df.S2q, np.linspace(0, 1e3, 101), histtype="step")
plt.xlabel("S2 charge (pes)")
plt.ylabel("Fraction of events (%)")
plt.grid()

df = None
plt.tight_layout()

In [ ]:
def same_s1_s2(df):
    return df.nS1 == df.nS2

In [ ]:
df = data
df = df.loc[same_s1_s2]; effs["n_xS12"] = df.event.nunique()

print(len(df), df.event.nunique(), df.event.nunique()/data.event.nunique())
data = df

In [ ]:
effs["run"] = run
effs = effs.to_frame().T.set_index("run").reset_index()

In [ ]:
if outputfile.exists():
    ans = ""
    while ans.lower() not in "yn".split():
        ans = input("Overwrite? ")
        if ans=="y":
            break
        else:
            raise KeyboardInterrupt()
with tb.open_file(outputfile, "w", filters=tb.Filters(complib="zlib", complevel=4)) as file:
    df_writer(file, data, "DST", "Events", compression="ZLIB4")
    df_writer(file, effs, "EFF", "data", compression="ZLIB4")

In [ ]:
!ptdump -v $outputfile